<a href="https://colab.research.google.com/github/awesomebruh03/DCCTN/blob/main/DCCTN_V_01(beta)V1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Loading Phase

This section Loads the necessary data. TIMIT Corps from `mfekadu/darpa-timit-acousticphonetic-continuous-speech`.

VoiceBank Demand from `chrisfilo/demand`

And Mozila Common Voice Bangla accured from mozila foundation and published on Kaggle under Profile `sajidullah03`. Published Dataset is untouched.


In [29]:
!pip install kagglehub

In [30]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
#kagglehub.login()


In [31]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

TIMIT_PATH = kagglehub.dataset_download('mfekadu/darpa-timit-acousticphonetic-continuous-speech')
DEMAND_PATH = kagglehub.dataset_download('chrisfilo/demand')
BN_CV_PATH = kagglehub.dataset_download('sajidullah03/common-voice-24-bn')

print('Data source import complete.')


Using Colab cache for faster access to the 'darpa-timit-acousticphonetic-continuous-speech' dataset.
Using Colab cache for faster access to the 'demand' dataset.
Data source import complete.


In [32]:
print(BN_CV_PATH)
print(DEMAND_PATH)
print(BN_CV_PATH)

/root/.cache/kagglehub/datasets/sajidullah03/common-voice-24-bn/versions/1
/kaggle/input/demand
/root/.cache/kagglehub/datasets/sajidullah03/common-voice-24-bn/versions/1


In [33]:
import os
import random
import math
import torch
import torchaudio
from pathlib import Path

# Define the paths from your kagglehub output
#TIMIT_PATH = mfekadu_darpa_timit_acousticphonetic_continuous_speech_path
#DEMAND_PATH = "/root/.cache/kagglehub/datasets/chrisfilo/demand/versions/1"
#BN_CV_PATH = "/root/.cache/kagglehub/datasets/sajidullah03/common-voice-24-bn/versions/1"

# Target sample rate for Cochlear Implants (usually 16kHz is standard)
TARGET_SR = 16000

# **Utility Functions for Audio Mixing**

In [34]:
def calculate_power(tensor):
    """Calculates the power of an audio tensor."""
    return torch.mean(tensor ** 2)

def mix_audio(clean_speech, noise, target_snr_db):
    """
    Mixes clean speech and noise at a specific SNR.
    """
    # 1. Match the lengths. If noise is shorter, repeat it. If longer, crop it.
    if noise.shape[1] < clean_speech.shape[1]:
        repeats = math.ceil(clean_speech.shape[1] / noise.shape[1])
        noise = noise.repeat(1, repeats)

    # Randomly crop the noise to match the exact length of the speech
    start_idx = random.randint(0, noise.shape[1] - clean_speech.shape[1])
    noise = noise[:, start_idx : start_idx + clean_speech.shape[1]]

    # 2. Calculate powers
    clean_power = calculate_power(clean_speech)
    noise_power = calculate_power(noise)

    # Handle edge case of pure silence
    if clean_power == 0 or noise_power == 0:
        return clean_speech

    # 3. Calculate the multiplier to achieve the target SNR
    # SNR = 10 * log10(Power_S / Power_N)
    snr_linear = 10 ** (target_snr_db / 10.0)
    noise_multiplier = torch.sqrt(clean_power / (noise_power * snr_linear))

    # 4. Mix them
    scaled_noise = noise * noise_multiplier
    mixed_audio = clean_speech + scaled_noise

    # Prevent clipping (audio going above 1.0 or below -1.0)
    max_amp = torch.max(torch.abs(mixed_audio))
    if max_amp > 1.0:
        mixed_audio = mixed_audio / max_amp
        clean_speech = clean_speech / max_amp # scale clean speech the same way so the target is valid

    return mixed_audio, clean_speech

# **PyTorch Dataset Class**

In [35]:
class CIDenoisingDataset(torch.utils.data.Dataset):
    def __init__(self, clean_speech_dirs, noise_dir, snr_range=(-5, 10), target_sr=16000, segment_length_s=3.0):
        self.snr_range = snr_range
        self.target_sr = target_sr
        # Standardize audio chunks to exactly 3 seconds (helps with batching)
        self.target_length = int(target_sr * segment_length_s)

        # 1. Gather all clean speech files (.wav for TIMIT, .mp3 for Common Voice)
        self.clean_files = []
        for d in clean_speech_dirs:
            # Recursively find common audio extensions
            for ext in ['**/*.wav', '**/*.mp3', '**/*.WAV', '**/*.flac']:
                self.clean_files.extend(list(Path(d).rglob(ext)))

        # 2. Gather all noise files (.wav for DEMAND)
        self.noise_files = list(Path(noise_dir).rglob('**/*.wav'))

        print(f"Found {len(self.clean_files)} clean speech files.")
        print(f"Found {len(self.noise_files)} noise files.")

    def __len__(self):
        return len(self.clean_files)

    def _load_and_resample(self, filepath):
        waveform, sr = torchaudio.load(filepath)
        # Convert stereo to mono if necessary
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)
        # Resample if it doesn't match target
        if sr != self.target_sr:
            resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=self.target_sr)
            waveform = resampler(waveform)
        return waveform

    def _pad_or_crop(self, waveform):
        """Ensures all audio snippets are exactly the same length."""
        if waveform.shape[1] < self.target_length:
            # Pad with silence
            padding = self.target_length - waveform.shape[1]
            waveform = torch.nn.functional.pad(waveform, (0, padding))
        elif waveform.shape[1] > self.target_length:
            # Randomly crop
            start = random.randint(0, waveform.shape[1] - self.target_length)
            waveform = waveform[:, start:start + self.target_length]
        return waveform

    def __getitem__(self, idx):
        # 1. Load clean speech
        clean_path = self.clean_files[idx]
        clean_speech = self._load_and_resample(clean_path)
        clean_speech = self._pad_or_crop(clean_speech)

        # 2. Randomly select and load a noise file
        noise_path = random.choice(self.noise_files)
        noise = self._load_and_resample(noise_path)

        # ---> NEW FIX: Ensure noise is the exact same length as clean_speech!
        noise = self._pad_or_crop(noise)

        # 3. Randomly select an SNR from your target range (-5 to 10)
        target_snr = random.uniform(self.snr_range[0], self.snr_range[1])

        ## 4. Mix them!
        mixed_output = mix_audio(clean_speech, noise, target_snr)

        # --- NEW FIX: Bulletproof Type Conversion ---
        # If the mix_audio function returned a tuple or list (e.g., [tensor] or (tensor,) ),
        # extract the actual audio from the first position
        if isinstance(mixed_output, (tuple, list)):
            mixed_audio = mixed_output[0]
        else:
            mixed_audio = mixed_output

        # Force it to be a pure PyTorch FloatTensor
        if not isinstance(mixed_audio, torch.Tensor):
            mixed_audio = torch.tensor(mixed_audio, dtype=torch.float32)
        else:
            mixed_audio = mixed_audio.float()
        # --------------------------------------------

        # Returns the input for the network (mixed) and the ground-truth label (clean_speech)
        return mixed_audio, clean_speech

# **Test Section**

In [36]:
# Initialize the dataset
dataset = CIDenoisingDataset(
    clean_speech_dirs=[TIMIT_PATH, BN_CV_PATH],
    noise_dir=DEMAND_PATH,
    snr_range=(-5, 10),
    segment_length_s=3.0 # 3 seconds of audio per sample
)

# Initialize a DataLoader for batching
dataloader = torch.utils.data.DataLoader(dataset, batch_size=4, shuffle=True)

# Test fetching one batch
for mixed_batch, clean_batch in dataloader:
    print(f"Mixed audio batch shape: {mixed_batch.shape} (Batch, Channel, Time)")
    print(f"Clean audio batch shape: {clean_batch.shape}")
    break # Just test the first batch

Found 1064690 clean speech files.
Found 560 noise files.
Mixed audio batch shape: torch.Size([4, 1, 48000]) (Batch, Channel, Time)
Clean audio batch shape: torch.Size([4, 1, 48000])


# **The Complex STFT Module**

In [37]:
import torch
import torch.nn as nn

class ComplexSTFT(nn.Module):
    def __init__(self, n_fft=512, hop_length=256, win_length=512):
        super().__init__()
        self.n_fft = n_fft
        self.hop_length = hop_length
        self.win_length = win_length
        # We use a Hann window to smooth the edges of our audio chunks
        # Registering it as a buffer ensures it moves to the GPU automatically if your model does
        self.register_buffer('window', torch.hann_window(win_length))

    def forward(self, waveform):
        """
        Converts 1D audio waveform into 2D Complex STFT tensor.
        Input: (Batch, 1, Time)
        Output: (Batch, 2, Freq, Frames) -> Channel 0 is Real, Channel 1 is Imaginary
        """
        # Remove the channel dimension for the STFT function
        # Shape becomes (Batch, Time)
        waveform = waveform.squeeze(1)

        # Compute STFT
        # return_complex=True is the standard in modern PyTorch
        stft_complex = torch.stft(
            waveform,
            n_fft=self.n_fft,
            hop_length=self.hop_length,
            win_length=self.win_length,
            window=self.window,
            center=True,
            return_complex=True
        )

        # stft_complex shape: (Batch, Freq, Frames)
        # We separate it into real and imaginary tensors
        real_part = stft_complex.real
        imag_part = stft_complex.imag

        # Stack them along a new channel dimension
        # Output shape: (Batch, 2, Freq, Frames)
        stft_tensor = torch.stack([real_part, imag_part], dim=1)

        return stft_tensor

    def inverse(self, stft_tensor):
        """
        Converts 2D Complex STFT tensor back into 1D audio waveform.
        Input: (Batch, 2, Freq, Frames)
        Output: (Batch, 1, Time)
        """
        # Separate the real and imaginary channels
        real_part = stft_tensor[:, 0, :, :]
        imag_part = stft_tensor[:, 1, :, :]

        # Recombine into PyTorch complex tensor
        stft_complex = torch.complex(real_part, imag_part)

        # Compute Inverse STFT
        waveform = torch.istft(
            stft_complex,
            n_fft=self.n_fft,
            hop_length=self.hop_length,
            win_length=self.win_length,
            window=self.window,
            center=True
        )

        # Add the channel dimension back: (Batch, 1, Time)
        return waveform.unsqueeze(1)

# **Test the Transformation**

In [38]:
# Initialize our STFT module
# n_fft=512 and hop_length=256 are standard for 16kHz audio.
# hop_length=256 means we have a step size of 16ms, which is great for CI processing.
stft_module = ComplexSTFT(n_fft=512, hop_length=256)

# 1. Test the Forward pass (Waveform -> STFT)
stft_features = stft_module(mixed_batch)
print(f"STFT Tensor shape: {stft_features.shape} (Batch, Real/Imag, Freq_Bins, Time_Frames)")

# 2. Test the Inverse pass (STFT -> Waveform)
reconstructed_audio = stft_module.inverse(stft_features)
print(f"Reconstructed Audio shape: {reconstructed_audio.shape}")

# 3. Verify we didn't lose any data in the process (Perfect Reconstruction)
# The difference between original and reconstructed should be practically zero
# Note: ISTFT can sometimes produce a slightly different length than the original.
# For comparison, we crop the original mixed_batch to match the reconstructed length.
original_length = mixed_batch.shape[-1]
reconstructed_length = reconstructed_audio.shape[-1]

# Crop mixed_batch to match the reconstructed_audio length for comparison
# This ensures element-wise operations can be performed.
cropped_mixed_batch = mixed_batch[:, :, :reconstructed_length]

difference = torch.max(torch.abs(cropped_mixed_batch - reconstructed_audio))
print(f"Maximum reconstruction error (on common length): {difference.item():.6f}")
if original_length != reconstructed_length:
    print(f"Warning: Original audio length ({original_length}) differs from reconstructed length ({reconstructed_length}).")
    print("To achieve exact length matching, the `istft` function in `ComplexSTFT` would need the `length` parameter set to the original waveform length.")

STFT Tensor shape: torch.Size([4, 2, 257, 188]) (Batch, Real/Imag, Freq_Bins, Time_Frames)
Reconstructed Audio shape: torch.Size([4, 1, 47872])
Maximum reconstruction error (on common length): 0.000000
To achieve exact length matching, the `istft` function in `ComplexSTFT` would need the `length` parameter set to the original waveform length.


# **Generating the Causal Mask**

In [39]:
import torch
import torch.nn as nn

def generate_causal_mask(seq_len):
    """
    Generates a causal mask for the Transformer.
    Positions with True (or 0) are allowed to attend.
    Positions with False (or -inf) are masked out (the future).
    """
    # Create an upper triangular matrix of negative infinity
    mask = (torch.triu(torch.ones(seq_len, seq_len)) == 1).transpose(0, 1)
    mask = mask.float().masked_fill(mask == 0, float('-inf')).masked_fill(mask == 1, float(0.0))
    return mask

# **The Causal Transformer Bottleneck**

In [40]:
class CausalTransformerBottleneck(nn.Module):
    def __init__(self, d_model, num_heads, num_layers, dim_feedforward):
        super().__init__()
        self.d_model = d_model

        # Standard Transformer Encoder Layer
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=num_heads,
            dim_feedforward=dim_feedforward,
            batch_first=True, # Input format: (Batch, Seq_Len, Features)
            norm_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

    def forward(self, x):
        # x shape from encoder: (Batch, Channels, Freq, Time)
        b, c, f, t = x.shape

        # 1. Reshape for Transformer
        # We want Time to be the Sequence Length.
        # So we collapse Channels and Freq into a single feature dimension.
        # Shape becomes: (Batch, Channels * Freq, Time)
        x = x.view(b, c * f, t)

        # Swap Time and Feature dimensions to match batch_first=True
        # Shape becomes: (Batch, Time, Features)
        x = x.transpose(1, 2)

        # 2. Generate Causal Mask dynamically based on current time steps
        causal_mask = generate_causal_mask(t).to(x.device)

        # 3. Pass through Transformer with the causal mask
        # is_causal=True is a PyTorch optimization hint
        x = self.transformer(x, mask=causal_mask, is_causal=True)

        # 4. Reshape back to image format for the Decoder
        x = x.transpose(1, 2)            # (Batch, Features, Time)
        x = x.view(b, c, f, t)           # (Batch, Channels, Freq, Time)

        return x

# **The UNET**

In [41]:
class CI_ComplexTransformerUNet(nn.Module):
    def __init__(self):
        super().__init__()

        # ENCODER: Compressing the 257 frequencies
        # Input: (Batch, 2, 257, Time)
        self.enc1 = nn.Sequential(
            # kernel=(5, 3), stride=(2, 1) -> Halves frequency, keeps time
            nn.Conv2d(in_channels=2, out_channels=16, kernel_size=(5, 3), stride=(2, 1), padding=(2, 1)),
            nn.BatchNorm2d(16),
            nn.LeakyReLU(0.2)
        ) # Output: (Batch, 16, 129, Time)

        self.enc2 = nn.Sequential(
            nn.Conv2d(in_channels=16, out_channels=32, kernel_size=(5, 3), stride=(2, 1), padding=(2, 1)),
            nn.BatchNorm2d(32),
            nn.LeakyReLU(0.2)
        ) # Output: (Batch, 32, 65, Time)

        # TRANSFORMER BOTTLENECK
        # Features going in: Channels(32) * Freq(65) = 2080 feature dimension
        self.bottleneck = CausalTransformerBottleneck(
            d_model=2080,
            num_heads=8,
            num_layers=2, # Keep it shallow for CI battery life / latency
            dim_feedforward=2048
        )

        # DECODER: Expanding back to 257 frequencies
        self.dec2 = nn.Sequential(
            nn.ConvTranspose2d(32, 16, kernel_size=(5, 3), stride=(2, 1), padding=(2, 1), output_padding=(0, 0)),
            nn.BatchNorm2d(16),
            nn.LeakyReLU(0.2)
        ) # Output: (Batch, 16, 129, Time)

        self.dec1 = nn.Sequential(
            nn.ConvTranspose2d(16, 2, kernel_size=(5, 3), stride=(2, 1), padding=(2, 1), output_padding=(0, 0)),
        ) # Output: (Batch, 2, 257, Time)

    def forward(self, x):
        # Encoder
        e1 = self.enc1(x)
        e2 = self.enc2(e1)

        # Transformer
        b = self.bottleneck(e2)

        # Decoder (Notice we could add skip connections here: b + e2)
        d2 = self.dec2(b)

        # Match dimensions exactly before the final layer (STFT math requires precision)
        if d2.shape[2] != e1.shape[2]:
            d2 = d2[:, :, :e1.shape[2], :]

        d1 = self.dec1(d2)

        if d1.shape[2] != x.shape[2]:
            d1 = d1[:, :, :x.shape[2], :]

        return d1

In [42]:
# Initialize the model
model = CI_ComplexTransformerUNet()

# Dummy tensor simulating your Phase 2 output
phase2_output = torch.randn(4, 2, 257, 188)

# Pass it through the model
enhanced_stft = model(phase2_output)

print(f"Input shape: {phase2_output.shape}")
print(f"Output shape: {enhanced_stft.shape}")

/tmp/ipykernel_796/1589505924.py:14: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)


Input shape: torch.Size([4, 2, 257, 188])
Output shape: torch.Size([4, 2, 257, 188])


# **Define the Composite Loss Function**

In [43]:
import torch
import torch.nn as nn
import torch.nn.functional as F

def calculate_sisdr(estimate, reference, eps=1e-8):
    """
    Calculates Scale-Invariant Signal-to-Distortion Ratio (SI-SDR).
    Higher is better, so we will return the negative value for the loss.
    """
    # Remove mean to ensure scale invariance
    estimate = estimate - torch.mean(estimate, dim=-1, keepdim=True)
    reference = reference - torch.mean(reference, dim=-1, keepdim=True)

    # Calculate scale factor
    dot_product = torch.sum(reference * estimate, dim=-1, keepdim=True)
    ref_energy = torch.sum(reference ** 2, dim=-1, keepdim=True) + eps
    alpha = dot_product / ref_energy

    # Target and error signals
    target = alpha * reference
    error = estimate - target

    # Calculate SI-SDR
    target_energy = torch.sum(target ** 2, dim=-1) + eps
    error_energy = torch.sum(error ** 2, dim=-1) + eps
    sisdr = 10 * torch.log10(target_energy / error_energy)

    return sisdr.mean()

class CI_CompositeLoss(nn.Module):
    def __init__(self, alpha=0.1):
        super().__init__()
        self.alpha = alpha # Weighting factor for the spectral loss

    def forward(self, est_audio, clean_audio, est_stft, clean_stft):
        # 1. Time-Domain Loss (Negative SI-SDR because we want to minimize the loss)
        # est_audio and clean_audio shape: (Batch, 1, Time)
        loss_sisdr = -calculate_sisdr(est_audio.squeeze(1), clean_audio.squeeze(1))

        # 2. Frequency-Domain Loss (Magnitude MSE)
        # Calculate magnitude: sqrt(real^2 + imag^2)
        est_mag = torch.sqrt(est_stft[:, 0, :, :]**2 + est_stft[:, 1, :, :]**2 + 1e-8)
        clean_mag = torch.sqrt(clean_stft[:, 0, :, :]**2 + clean_stft[:, 1, :, :]**2 + 1e-8)

        loss_spectral = F.mse_loss(est_mag, clean_mag)

        # 3. Combine them
        total_loss = loss_sisdr + (self.alpha * loss_spectral)
        return total_loss, loss_sisdr, loss_spectral

# **Checkpoint Saving and loading Utility function**

In [44]:
from google.colab import drive
import os

# This will prompt you to log in and grant Colab access to your Drive
drive.mount('/content/drive')

# Create a dedicated folder in your Drive for your Capstone Project
# This ensures your Drive doesn't get cluttered
project_dir = '/content/drive/MyDrive/CI_Capstone_Project'
os.makedirs(project_dir, exist_ok=True)

# Define the absolute path where the checkpoint will live on your Drive
checkpoint_path = os.path.join(project_dir, 'ci_model_checkpoint.pth')

print(f"All checkpoints will be safely saved to: {checkpoint_path}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
All checkpoints will be safely saved to: /content/drive/MyDrive/CI_Capstone_Project/ci_model_checkpoint.pth


In [45]:
def save_checkpoint(model, optimizer, scheduler, epoch, best_loss, filename="ci_model_checkpoint.pth"):
    """Saves the entire training state to a file."""
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'best_loss': best_loss
    }
    torch.save(checkpoint, filename)
    print(f"✅ Checkpoint saved: Epoch {epoch} at {filename}")

def load_checkpoint(filepath, model, optimizer, scheduler):
    """Loads the training state to resume training."""
    if os.path.isfile(filepath):
        checkpoint = torch.load(filepath, map_location=torch.device('cuda' if torch.cuda.is_available() else 'cpu'))
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        start_epoch = checkpoint['epoch'] + 1
        best_loss = checkpoint['best_loss']
        print(f"🔄 Resuming training from Epoch {start_epoch}")
        return start_epoch, best_loss
    else:
        print("⚠️ No checkpoint found. Starting from scratch.")
        return 0, float('inf')

In [52]:
# Initialize the dataset
dataset = CIDenoisingDataset(
    clean_speech_dirs=[TIMIT_PATH, BN_CV_PATH],
    noise_dir=DEMAND_PATH,
    snr_range=(-5, 10),
    segment_length_s=3.0 # 3 seconds of audio per sample
)

# ---> NEW FIX: The High-Speed DataLoader <---
train_dataloader = torch.utils.data.DataLoader(
    dataset,
    batch_size=32,       # Increased from 4. Feeds 32 files to the GPU at once!
    shuffle=True,
    num_workers=4,       # Uses 2 background CPU cores to prepare the next batch
    pin_memory=True,     # Unlocks a fast-lane for data moving from CPU RAM to GPU VRAM
    drop_last=True       # Prevents crashes if the last batch is smaller than 32
)

# NOTE: If you have a val_dataloader right below this, update it with batch_size=32, num_workers=2, and pin_memory=True as well!

Found 1064690 clean speech files.
Found 560 noise files.


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


# **The Training Loop**

In [53]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
model = model.to(device)
stft_module = stft_module.to(device)

cuda


In [ ]:
from tqdm.auto import tqdm

# Setup Optimizer and Scheduler
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5)
criterion = CI_CompositeLoss(alpha=100.0)

stft_module = stft_module.to(device)

# Early Stopping Parameters
patience = 7
epochs_no_improve = 0

# Attempt to load from a previous Colab session
start_epoch, best_val_loss = load_checkpoint(checkpoint_path, model, optimizer, scheduler)
total_epochs = 5

for epoch in range(start_epoch, total_epochs):
    # ========================
    # 1. TRAINING PHASE
    # ========================
    model.train()
    train_loss = 0.0

    # Wrap your dataloader in tqdm for a progress bar
    train_pbar = tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{total_epochs} [Train]", leave=False)

    for batch_idx, (mixed_audio, clean_audio) in enumerate(train_pbar):
        mixed_audio, clean_audio = mixed_audio.to(device), clean_audio.to(device)
        optimizer.zero_grad()

        # Forward Pass
        mixed_stft = stft_module(mixed_audio)
        clean_stft = stft_module(clean_audio)
        est_stft = model(mixed_stft)

        # Inverse & Length Match
        est_audio = stft_module.inverse(est_stft)
        min_len = min(est_audio.shape[-1], clean_audio.shape[-1])
        est_audio, clean_audio = est_audio[:, :, :min_len], clean_audio[:, :, :min_len]

        # Loss & Backprop
        loss, sisdr, spectral = criterion(est_audio, clean_audio, est_stft, clean_stft)
        loss.backward()

        # Gradient Clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        train_loss += loss.item()

        # Update the progress bar text with the live loss and SI-SDR
        train_pbar.set_postfix({'Loss': f"{loss.item():.4f}", 'SI-SDR': f"{-sisdr.item():.2f}dB"})

    avg_train_loss = train_loss / len(train_dataloader)

    # ========================
    # 2. VALIDATION PHASE
    # ========================
    model.eval()
    val_loss = 0.0

    # Wrap validation loader in tqdm as well
    val_pbar = tqdm(val_dataloader, desc=f"Epoch {epoch+1}/{total_epochs} [Val]", leave=False)

    with torch.no_grad():
        for mixed_audio, clean_audio in val_pbar:
            mixed_audio, clean_audio = mixed_audio.to(device), clean_audio.to(device)

            mixed_stft = stft_module(mixed_audio)
            clean_stft = stft_module(clean_audio)
            est_stft = model(mixed_stft)

            est_audio = stft_module.inverse(est_stft)
            min_len = min(est_audio.shape[-1], clean_audio.shape[-1])
            est_audio, clean_audio = est_audio[:, :, :min_len], clean_audio[:, :, :min_len]

            loss, sisdr, spectral = criterion(est_audio, clean_audio, est_stft, clean_stft)
            val_loss += loss.item()

            # Live updates for validation
            val_pbar.set_postfix({'Loss': f"{loss.item():.4f}", 'SI-SDR': f"{-sisdr.item():.2f}dB"})

    avg_val_loss = val_loss / len(val_dataloader)

    # Print the final summary for the epoch so it stays on the screen
    print(f"✅ Epoch {epoch+1} Summary | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

    # ========================
    # 3. OVERFITTING COUNTERMEASURES & CHECKPOINTING
    # ========================
    scheduler.step(avg_val_loss)

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        epochs_no_improve = 0
        save_checkpoint(model, optimizer, scheduler, epoch, best_val_loss, checkpoint_path)
    else:
        epochs_no_improve += 1
        print(f"⚠️ No improvement in validation loss for {epochs_no_improve} epochs.")

        if epochs_no_improve >= patience:
            print("🛑 Early stopping triggered! The model has started overfitting.")
            break

⚠️ No checkpoint found. Starting from scratch.


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Epoch 1/5 [Train]:   0%|          | 0/33271 [00:00<?, ?it/s]